In [0]:
%pip install lightgbm xgboost
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "olist_dev")
CATALOG = dbutils.widgets.get("catalog")
TABLE_FEATURES = f"{CATALOG}.ml.features_commandes"
NOM_MODELE = f"{CATALOG}.ml.delivery_delay_model"
TABLE_PREDICTIONS = f"{CATALOG}.lakehouse.gold_delivery_predictions"
print(TABLE_FEATURES, NOM_MODELE, TABLE_PREDICTIONS, sep="\n")

In [0]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F
from olist.bronze import ecrire_table
from olist.scoring import ALIAS_CHAMPION, commandes_a_scorer, predire

mlflow.set_registry_uri("databricks-uc")

In [0]:
client = MlflowClient()
version = client.get_model_version_by_alias(NOM_MODELE, ALIAS_CHAMPION)
seuil = float(version.tags["seuil"])
pipeline = mlflow.sklearn.load_model(f"models:/{NOM_MODELE}@{ALIAS_CHAMPION}")
print(f"champion : v{version.version} ({version.tags.get('algorithme')}), seuil {seuil:.3f}")

In [0]:
commandes = commandes_a_scorer(spark, TABLE_FEATURES)
pred = predire(pipeline, commandes, seuil)

predictions = (
    spark.createDataFrame(pred)
    .withColumn("seuil", F.lit(seuil))
    .withColumn("version_modele", F.lit(int(version.version)))
    .withColumn("date_scoring", F.current_timestamp())
)
ecrire_table(predictions, TABLE_PREDICTIONS)
print(f"{len(pred):,} prédictions écrites dans {TABLE_PREDICTIONS}")

In [0]:
p = spark.table(TABLE_PREDICTIONS)
attendu = spark.table(TABLE_FEATURES).filter("order_approved_at IS NOT NULL").count()
erreurs = []

n = p.count()
if n != attendu:
    erreurs.append(f"{n:,} prédictions au lieu de {attendu:,}")
if p.select("order_id").distinct().count() != n:
    erreurs.append("order_id non unique")
if p.filter("proba_retard < 0 OR proba_retard > 1 OR proba_retard IS NULL").count():
    erreurs.append("probabilités hors de [0, 1] ou nulles")

display(
    p.groupBy("periode")
    .agg(F.count("*").alias("commandes"),
         F.round(F.avg("proba_retard"), 3).alias("proba_moyenne"),
         F.round(F.avg(F.col("alerte").cast("int")) * 100, 1).alias("taux_alertes_pct"))
    .orderBy("periode")
)
assert not erreurs, erreurs

In [0]:
from sklearn.metrics import average_precision_score

verif = (
    p.join(spark.table(TABLE_FEATURES).select("order_id", "en_retard", "dans_population"), "order_id")
    .filter("dans_population AND periode = 'test'")
    .select("en_retard", "proba_retard")
    .toPandas()
)
pr_auc_scoring = average_precision_score(verif["en_retard"], verif["proba_retard"])
pr_auc_train = client.get_run(version.run_id).data.metrics["test_pr_auc"]

print(f"PR-AUC test, mesurée dans 05_train : {pr_auc_train:.4f}")
print(f"PR-AUC test, recalculée ici        : {pr_auc_scoring:.4f}")
assert abs(pr_auc_scoring - pr_auc_train) < 1e-6, "le scoring ne reproduit pas l'entraînement"